# Reading the Inference Report

This example is part of the *Universal Inference* section. With `fallback = LocalLaplace()`, RxInfer uses exact rules where a model has them and approximates the other messages and posteriors. The posteriors themselves do not show which parts were approximated, or how. To record this, pass `report = true` to `infer`; it works with all of the fallback schemes. The result then has a report, `result.report`, with two sections:

- `result.report.approximations` lists the approximated posteriors and how each was projected. For each posterior it lists the node instances that sent messages to it, counts how many of those messages came from an exact rule and how many from the rule fallback, and names the backends that computed the fallback's expectations. It also warns about the parts of the inference that are most likely to be inaccurate.
- `result.report.timings` shows how the inference time was spent.

This notebook reads the report of one model under two inference schemes, and then uses the report as data.

In [ ]:
using RxInfer, StatsFuns, StableRNGs, PrettyTables

## The model

A logistic regression with one coefficient $\beta$, whose prior precision $\tau$ is itself unknown:

$$\begin{aligned}
\tau &\sim \mathrm{Gamma}(2, 1),\\
\beta &\sim \mathcal{N}(0, \tau^{-1}),\\
y_i &\sim \mathrm{Bernoulli}\left(\sigma(\beta x_i)\right).
\end{aligned}$$

The Gamma prior on the precision of a Gaussian is conjugate under mean field, so RxInfer has exact variational rules for $\tau$. The likelihood has no rules. We define it as a custom node that has a log-density and nothing else.

In [ ]:
rng = StableRNG(3)
n = 60
x = randn(rng, n)
β_true = 1.5
y = [rand(rng) < logistic(β_true * xi) ? 1.0 : 0.0 for xi in x]

struct LogisticBernoulli{B, X} <: DiscreteUnivariateDistribution
    β::B
    x::X
end

BayesBase.logpdf(d::LogisticBernoulli, y) = y * log(logistic(d.β * d.x)) + (1 - y) * log1p(-logistic(d.β * d.x))
BayesBase.insupport(::LogisticBernoulli, y) = true

@define_factor_node(node = LogisticBernoulli, type = Stochastic, interfaces = [:out, :β, :x])

@model function shrinkage(y, x)
    τ ~ Gamma(shape = 2.0, rate = 1.0)
    β ~ NormalMeanPrecision(0.0, τ)
    for i in eachindex(y)
        y[i] ~ LogisticBernoulli(β, x[i])
    end
end

## Under mean field

We factorise the posterior as $q(\beta)\,q(\tau)$ and ask for the report.

In [ ]:
mean_field = @constraints begin
    q(β, τ) = q(β)q(τ)
end

result = infer(
    model = shrinkage(x = x),
    data = (y = y,),
    constraints = mean_field,
    iterations = 10,
    fallback = LocalLaplace(),
    free_energy = true,
    report = true,
)

q_β, q_τ = result.posteriors[:β][end], result.posteriors[:τ][end]
@assert abs(mean(q_β) - β_true) < 2 * std(q_β) # hide

With PrettyTables.jl loaded, the report prints as tables.

In [ ]:
approximations = result.report.approximations # hide
@assert isempty(RxInfer.diagnostics(result.report)) # hide
@assert RxInfer.is_approximated(approximations.variables[(:β, nothing)]) # hide
@assert !RxInfer.is_approximated(approximations.variables[(:τ, nothing)]) # hide
result.report.approximations

The first line is a summary. One of the two posteriors was approximated. The posterior of $\tau$ is exact, because its Gamma posterior comes from the conjugate rules. There are no warnings.

In the table of approximated posteriors, each approximated posterior has a group of rows. The header of the group says how the posterior was projected: here by Laplace's method, with the number of Newton iterations and the log-normaliser. Each row is a node instance as it is written in the model. Instances of the same form share one row, as in `y[1:60] ~ LogisticBernoulli(β, …)`, where `…` stands for the inputs $x_i$, which differ between the instances.

The `exact` and `fallback` columns count the messages from a node towards the variable that came from an exact rule and from the rule fallback, over all ten iterations. The `expectations` column says how the fallback computed its expectations. Here they are exact, because under mean field the other inputs of the likelihood are the data and the constants $x_i$.

The table of joints and energies computed by the fallback lists the average energies of the likelihood, which the free energy needs. The last line lists the expectation backends that were available, in the order in which they are tried.

## Under belief propagation, a warning

Without the factorisation, inference is belief propagation. The Gaussian node $\beta \sim \mathcal{N}(0, \tau^{-1})$ then has no exact rule, because its precision is random, so the fallback computes its messages as well. The report flags this.

In [ ]:
result_bp = infer(
    model = shrinkage(x = x),
    data = (y = y,),
    fallback = LocalLaplace(),
    report = true,
)
found = RxInfer.diagnostics(result_bp.report)
@assert [d.kind for d in found] == [:expectation_over_messages] # hide
result_bp.report.approximations

The warning has the kind `:expectation_over_messages`. Under belief propagation, the fallback computes the message of the Gaussian node towards $\beta$ as an expectation over the *message* from $\tau$. This gives the variational message. The belief propagation message would instead integrate the node against the message from $\tau$. The result approximates belief propagation, and in this model the error of that approximation is not controlled. There are two ways to avoid it:

- Use mean field. With the factorisation above, the expectation is taken over a marginal, which is what variational message passing prescribes, and the warning goes away.
- Use a cavity method. Under `fallback = ExpectationPropagation()` or `LaplacePropagation()`, a node with several random inputs sends the belief propagation message, which integrates the node against the messages of the other inputs, but only when those messages are Gaussian. An input whose message is not Gaussian, such as the Gamma message of $\tau$ here, still enters through the expectation of the log-density. For this model, mean field is therefore the fix.

Each diagnostic is a value with a kind, a level, a subject and a message:

In [ ]:
found

A test, or a notebook such as this one, can check that an inference raised no warnings with `isempty(filter(d -> d.level === :warning, RxInfer.diagnostics(result.report)))`.

## The report as data

The tables above are a formatted view of data stored in the report, which code can read directly:
- `RxInfer.node_types(report)` counts the computations by node type;
- `report.variables` records, for each variable, its projections and the messages it received;
- `report_tables` returns the tables.

For example, the number of times the fallback computed a message from the likelihood towards $\beta$:

In [ ]:
RxInfer.node_types(result.report.approximations)[(LogisticBernoulli, :message, :β)].fallbacks

The last Laplace projection of $\beta$, as reported by `LaplaceForm`:

In [ ]:
result.report.approximations.variables[(:β, nothing)].marginal.outcome

The rows of the table of approximated posteriors:

In [ ]:
first(report_tables(result.report.approximations)).rows

## Timings

The second section of the report shows how the time was spent. Julia compiles a model the first time it runs, so we look at a second run.

In [ ]:
infer(model = shrinkage(x = x), data = (y = y,), constraints = mean_field, iterations = 10, fallback = LocalLaplace(), free_energy = true) # hide
timed = infer(
    model = shrinkage(x = x),
    data = (y = y,),
    constraints = mean_field,
    iterations = 10,
    fallback = LocalLaplace(),
    free_energy = true,
    report = (:timings,),
)
@assert length(timed.report.timings.iterations) == 10 # hide
timed.report

The time of a call does not include the time of the calls it makes, which is counted under their own kind, so the times of the kinds add up. The time not spent in rules, the fallback or projections is spent by the engine itself: on products of messages, on the streams between the nodes and on the entropies of the free energy. In this small model, the engine takes most of the time. The Laplace projections of $\beta$ and the quadrature of the fallback over the 60 observations take a few percent each, and the exact rules take almost no time. For timing statistics over many runs, use RxInfer's benchmark callbacks.

## Summary

- `report = true` records which posteriors an inference approximated, and for each one the node instances and rules that contributed. The report is off by default and costs nothing when off.
- `RxInfer.diagnostics` lists the parts of the inference most likely to be inaccurate. Here it flagged a fallback that averages over a message under belief propagation, which mean field avoids.
- The counts, projections and tables of the report are data that code can read and check.